## Overview
Welcome to the 2026 Kaggle Playground Series! We plan to continue in the spirit of previous playgrounds, providing interesting and approachable datasets for our community to practice their machine learning skills, and anticipate a competition each month.

Your Goal: Predict the stellar class.

## Evaluation
Submissions are evaluated on balanced accuracy between the predicted class and observed target.

## About Dataset
The dataset for this competition (both train and test) was inspired by the Stellar classification dataset. Feature distributions are close to, but not exactly the same, as the original.

- train.csv - the training set, with class as target
- test.csv - the test set, used to predict the category for class
- sample_submission.csv - a sample submission file in the correct format

## Importing Libraries | Data

In [8]:
import os
import math
import random
import warnings
import numpy as np, pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer
warnings.filterwarnings('ignore')

import torch
torch.set_num_threads(4)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.12.0+cpu
CUDA available: False


In [9]:
# Configurations
ROOT_PATH = "playground-series-s6e6"
BENCHMARK = 0.97
SEED = 42
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

Using device: cpu


In [10]:
train_df_org = pd.read_csv(os.path.join(ROOT_PATH, 'train.csv'))
test_df_org = pd.read_csv(os.path.join(ROOT_PATH, 'test.csv'))
sub_df = pd.read_csv(os.path.join(ROOT_PATH, 'sample_submission.csv'))

print(f"Shape of train: {train_df_org.shape} | test: {test_df_org.shape} | sub_df: {sub_df.shape}")

Shape of train: (577347, 12) | test: (247435, 11) | sub_df: (247435, 2)


## Feature Engineering

Key understanding of features:
- **alpha, delta**: Right Ascension & Declination in degrees (sky position)
- **u, g, r, i, z**: SDSS photometric magnitudes (ultraviolet→near-IR). Smaller = brighter
- **redshift**: Doppler shift of spectral lines — STAR≈0.07, GALAXY≈0.51, QSO≈1.88 (most discriminative!)
- **spectral_type**: Derived from `r-g` color (M=very red, G/K=orange/yellow, A/F=white, O/B=blue)
- **galaxy_population**: Derived from `u-r` color index (Red_Sequence vs Blue_Cloud)

Feature engineering strategy:
1. **Color indices** (magnitude differences): u-g, g-r, r-i, i-z, u-r, etc. — standard astronomical colors
2. **Galactic coordinates** from equatorial (alpha, delta) → galactic latitude/longitude
3. **Photometric statistics**: mean, std, range of magnitudes
4. **Redshift-based features**: log transform, binned
5. **Interaction features**: redshift × color, redshift × spectral category
6. **Joint category encoding**: spectral_type + galaxy_population
7. **Equatorial & galactic Cartesian coordinates** for the MLP

In [11]:
# Rotation matrix: Equatorial (J2000) → Galactic
# Reference: IAU convention
RGE = np.array([
    [-0.054875539, -0.873437105, -0.483834992],
    [ 0.494109454, -0.444829594,  0.746982249],
    [-0.867666136, -0.198076390,  0.455983795]
])

def feature_engineering(df):
    df = df.copy()

    # ── 1. All pairwise color indices (magnitude differences) ──────────────
    # Standard astronomical colors: differences between adjacent/non-adjacent bands
    df['u_g'] = df['u'] - df['g']   # UV-optical color
    df['g_r'] = df['g'] - df['r']   # blue-green color (key for spectral type)
    df['r_i'] = df['r'] - df['i']   # red color index
    df['i_z'] = df['i'] - df['z']   # near-IR index
    df['u_r'] = df['u'] - df['r']   # broad color (key for galaxy population)
    df['u_i'] = df['u'] - df['i']   # UV to red
    df['u_z'] = df['u'] - df['z']   # UV to near-IR
    df['g_i'] = df['g'] - df['i']   # optical color
    df['g_z'] = df['g'] - df['z']   # optical to near-IR
    df['r_z'] = df['r'] - df['z']   # red to near-IR

    # ── 2. Photometric statistics across bands ─────────────────────────────
    mags = df[['u', 'g', 'r', 'i', 'z']].values
    df['mag_mean'] = mags.mean(axis=1)
    df['mag_std']  = mags.std(axis=1)
    df['mag_max']  = mags.max(axis=1)
    df['mag_min']  = mags.min(axis=1)
    df['mag_range'] = df['mag_max'] - df['mag_min']

    # ── 3. Redshift features ────────────────────────────────────────────────
    # redshift is the most discriminative single feature!
    rs = df['redshift'].values
    df['log1p_redshift'] = np.log1p(np.clip(rs, 0, None))
    df['sqrt_redshift']  = np.sqrt(np.clip(rs, 0, None))
    df['redshift_sq']    = rs ** 2
    df['redshift_cube']  = np.clip(rs ** 3, -1e6, 1e6)

    # ── 4. Equatorial Cartesian coordinates ────────────────────────────────
    alpha_rad = np.radians(df['alpha'].values)
    delta_rad = np.radians(df['delta'].values)
    df['eq_x'] = np.cos(delta_rad) * np.cos(alpha_rad)
    df['eq_y'] = np.cos(delta_rad) * np.sin(alpha_rad)
    df['eq_z'] = np.sin(delta_rad)

    # ── 5. Galactic coordinates ─────────────────────────────────────────────
    eq_vecs = np.stack([df['eq_x'], df['eq_y'], df['eq_z']], axis=1)
    gal_vecs = eq_vecs @ RGE.T
    df['gal_x'] = gal_vecs[:, 0]
    df['gal_y'] = gal_vecs[:, 1]
    df['gal_z'] = gal_vecs[:, 2]
    df['gal_l'] = np.degrees(np.arctan2(gal_vecs[:, 1], gal_vecs[:, 0])) % 360
    df['gal_b'] = np.degrees(np.arcsin(np.clip(gal_vecs[:, 2], -1.0, 1.0)))
    df['gal_b_abs'] = np.abs(df['gal_b'])

    # ── 6. Interaction features: redshift × key colors ──────────────────────
    df['redshift_x_u_r'] = df['redshift'] * df['u_r']
    df['redshift_x_g_r'] = df['redshift'] * df['g_r']
    df['redshift_x_u_g'] = df['redshift'] * df['u_g']

    # ── 7. Joint categorical ─────────────────────────────────────────────────
    df['joint_cat'] = df['spectral_type'] + '_' + df['galaxy_population']

    return df

print("Running feature engineering...")
train_fe = feature_engineering(train_df_org)
test_fe  = feature_engineering(test_df_org)

# ── Define feature columns ──────────────────────────────────────────────────
CAT_COLS = ['spectral_type', 'galaxy_population', 'joint_cat']
DROP_COLS = ['id', 'class'] + CAT_COLS
NUM_COLS = [c for c in train_fe.columns if c not in DROP_COLS]
FEATURE_COLS = NUM_COLS + CAT_COLS

TARGET = 'class'
LABEL_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

X_all   = train_fe[FEATURE_COLS].copy()
y_all   = train_fe[TARGET].map(LABEL_MAP).values.astype(np.int64)
X_test  = test_fe[FEATURE_COLS].copy()

print(f"Feature count: {len(FEATURE_COLS)} ({len(NUM_COLS)} numerical + {len(CAT_COLS)} categorical)")
print(f"X_all shape: {X_all.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Class distribution: {pd.Series(y_all).value_counts().sort_index().to_dict()}")
print(f"Features: {FEATURE_COLS}")

Running feature engineering...
Feature count: 42 (39 numerical + 3 categorical)
X_all shape: (577347, 42)
X_test shape: (247435, 42)
Class distribution: {0: 377480, 1: 117143, 2: 82724}
Features: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'mag_mean', 'mag_std', 'mag_max', 'mag_min', 'mag_range', 'log1p_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_cube', 'eq_x', 'eq_y', 'eq_z', 'gal_x', 'gal_y', 'gal_z', 'gal_l', 'gal_b', 'gal_b_abs', 'redshift_x_u_r', 'redshift_x_g_r', 'redshift_x_u_g', 'spectral_type', 'galaxy_population', 'joint_cat']


## RealMLP Architecture

We implement RealMLP from scratch using the key architectural tricks from
"Better by Default: Strong Pre-Tuned MLPs and Boosted Trees on Tabular Data" (NeurIPS 2024):

1. **ScalingLayer**: Learnable per-feature scale (learns feature importance)
2. **NTPLinear**: Neural Tangent Parameterization linear layer (1/sqrt(fan_in) normalization)
3. **SELU activation**: Self-normalizing — maintains zero-mean, unit-variance through layers
4. **Robust Scale + Smooth Clip**: Better-than-StandardScaler preprocessing for tabular data
5. **Label smoothing (0.1)**: Prevents overconfident predictions
6. **Cosine annealing LR schedule** with different LRs for scale/weights/biases
7. **Early stopping** on validation loss (using best checkpoint)

In [12]:
# ═══════════════════════════════════════════════════════════════════
# RealMLP Preprocessing Pipeline
# ═══════════════════════════════════════════════════════════════════

class CustomOneHotEncoder(BaseEstimator, TransformerMixin):
    """One-hot encode categorical features, with binary features as ±1."""
    def fit(self, X, y=None):
        self.ordinal_enc_ = OrdinalEncoder(
            unknown_value=np.nan,
            encoded_missing_value=np.nan,
            handle_unknown='use_encoded_value'
        )
        self.ordinal_enc_.fit(X)
        self.cat_sizes_ = []
        for cat_arr in self.ordinal_enc_.categories_:
            has_nan = np.any([isinstance(v, float) and np.isnan(v) for v in cat_arr])
            self.cat_sizes_.append(len(cat_arr) - int(has_nan))
        return self

    def transform(self, X, y=None):
        x_enc = self.ordinal_enc_.transform(X)
        n_samples = x_enc.shape[0]
        out_arrs = []
        for i, cat_size in enumerate(self.cat_sizes_):
            column  = x_enc[:, i]
            idxs    = np.arange(n_samples)
            isnan   = np.isnan(column)
            out_arr = np.zeros((n_samples, cat_size))
            out_arr[idxs[~isnan], column[~isnan].astype(np.int64)] = 1.0
            if cat_size == 2:
                out_arr = out_arr[:, 0:1] - out_arr[:, 1:2]
            out_arrs.append(out_arr)
        return np.concatenate(out_arrs, axis=-1)


class CustomOneHotPipeline(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.tfm_ = ColumnTransformer(transformers=[
            ('categorical', CustomOneHotEncoder(),
             make_column_selector(dtype_include=["string", "object", "category"])),
            ('remaining', FunctionTransformer(lambda x: x),
             make_column_selector(dtype_exclude=["string", "object", "category"]))
        ]).fit(X)
        return self

    def transform(self, X, y=None):
        return self.tfm_.transform(pd.DataFrame(X))


class RobustScaleSmoothClip(BaseEstimator, TransformerMixin):
    """Robust scaling + smooth clipping (tanh-like). Handles constant features."""
    def fit(self, X, y=None):
        assert isinstance(X, np.ndarray)
        self.median_  = np.median(X, axis=0)
        q75 = np.quantile(X, 0.75, axis=0)
        q25 = np.quantile(X, 0.25, axis=0)
        iqr = q75 - q25
        zero_iqr = iqr == 0.0
        # fallback to min-max range when IQR is zero
        iqr[zero_iqr] = 0.5 * (X.max(axis=0)[zero_iqr] - X.min(axis=0)[zero_iqr])
        self.factors_ = 1.0 / (iqr + 1e-30)
        self.factors_[iqr == 0.0] = 0.0   # constant feature → zero
        return self

    def transform(self, X, y=None):
        x_scaled = self.factors_[None, :] * (X - self.median_[None, :])
        return x_scaled / np.sqrt(1 + (x_scaled / 3) ** 2)


def get_realmlp_pipeline():
    return Pipeline([
        ('one_hot', CustomOneHotPipeline()),
        ('rssc',    RobustScaleSmoothClip())
    ])

print("Preprocessing classes defined.")

Preprocessing classes defined.


In [13]:
# ═══════════════════════════════════════════════════════════════════
# RealMLP Neural Network Modules
# ═══════════════════════════════════════════════════════════════════

class ScalingLayer(nn.Module):
    """Learnable per-feature scale — lets the model learn feature importance."""
    def __init__(self, n_features: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.scale[None, :]


class NTPLinear(nn.Module):
    """Neural Tangent Parameterization linear layer.
    Normalizes by 1/sqrt(fan_in) for stable training with large LR.
    """
    def __init__(self, in_features: int, out_features: int, zero_init: bool = False):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        factor = 0.0 if zero_init else 1.0
        self.weight = nn.Parameter(factor * torch.randn(in_features, out_features))
        self.bias   = nn.Parameter(factor * torch.randn(1, out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (1.0 / math.sqrt(self.in_features)) * (x @ self.weight) + self.bias


class Mish(nn.Module):
    """Smooth self-regularized non-monotonic activation (used for regression)."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.mul(torch.tanh(F.softplus(x)))


class RealMLP(nn.Module):
    """RealMLP: ScalingLayer → (NTPLinear → SELU) × n_layers → NTPLinear (zero-init).
    
    Key insight: SELU activation maintains normalised activations through deep networks.
    NTP parameterisation + separate LRs for scale/weights/biases enables large stable LR.
    """
    def __init__(self, input_dim: int, output_dim: int,
                 hidden_dim: int = 512, n_layers: int = 4,
                 dropout: float = 0.0):
        super().__init__()
        self.scaling = ScalingLayer(input_dim)
        layers = []
        in_dim = input_dim
        for i in range(n_layers):
            layers.append(NTPLinear(in_dim, hidden_dim))
            layers.append(nn.SELU())
            if dropout > 0:
                layers.append(nn.AlphaDropout(p=dropout))
            in_dim = hidden_dim
        layers.append(NTPLinear(in_dim, output_dim, zero_init=True))
        self.body = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(self.scaling(x))

    def get_param_groups(self):
        """Return parameter groups with different LR multipliers.
        Scale: 6x | Weights: 1x | Biases: 0.1x — per RealMLP paper defaults.
        """
        scale_params = list(self.scaling.parameters())
        weight_params, bias_params = [], []
        for module in self.body.modules():
            if isinstance(module, NTPLinear):
                weight_params.append(module.weight)
                bias_params.append(module.bias)
        return scale_params, weight_params, bias_params


print("RealMLP model class defined.")

RealMLP model class defined.


In [14]:
# ═══════════════════════════════════════════════════════════════════
# RealMLP Trainer
# ═══════════════════════════════════════════════════════════════════

class RealMLPTrainer:
    """Encapsulates training, validation, and prediction for RealMLP.
    
    Hyperparameters follow RealMLP-TD (Tuned Default) paper:
    - base_lr = 0.04, LR schedule: cosine with logarithmic time scale
    - Scale LR = 6×, Weight LR = 1×, Bias LR = 0.1×
    - Adam with β=(0.9, 0.95)
    - Label smoothing = 0.1
    - Batch size = 512 (large enough for full dataset)
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_dim: int = 512,
        n_layers: int = 4,
        n_epochs: int = 256,
        batch_size: int = 2048,
        base_lr: float = 0.04,
        label_smoothing: float = 0.1,
        dropout: float = 0.0,
        device: str = 'cpu',
        verbose: bool = True,
        patience: int = 32,
    ):
        self.input_dim       = input_dim
        self.output_dim      = output_dim
        self.hidden_dim      = hidden_dim
        self.n_layers        = n_layers
        self.n_epochs        = n_epochs
        self.batch_size      = batch_size
        self.base_lr         = base_lr
        self.label_smoothing = label_smoothing
        self.dropout         = dropout
        self.device          = device
        self.verbose         = verbose
        self.patience        = patience

    def _build_model(self):
        model = RealMLP(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            hidden_dim=self.hidden_dim,
            n_layers=self.n_layers,
            dropout=self.dropout,
        ).to(self.device)
        return model

    def _build_optimizer(self, model):
        scale_params, weight_params, bias_params = model.get_param_groups()
        return torch.optim.Adam(
            [
                {'params': scale_params,  'lr': 6.0   * self.base_lr},
                {'params': weight_params, 'lr': 1.0   * self.base_lr},
                {'params': bias_params,   'lr': 0.1   * self.base_lr},
            ],
            betas=(0.9, 0.95),
        )

    def _lr_schedule(self, t: float) -> float:
        """Cosine schedule on a log-time axis (RealMLP default)."""
        return 0.5 - 0.5 * np.cos(2 * np.pi * np.log2(1 + 15 * t))

    def fit(self, X_train: np.ndarray, y_train: np.ndarray,
            X_val: np.ndarray, y_val: np.ndarray):
        """Train model with early stopping based on validation CE loss."""
        model    = self._build_model()
        opt      = self._build_optimizer(model)
        criterion = nn.CrossEntropyLoss(label_smoothing=self.label_smoothing)

        x_tr = torch.as_tensor(X_train, dtype=torch.float32)
        y_tr = torch.as_tensor(y_train, dtype=torch.int64)
        x_va = torch.as_tensor(X_val,   dtype=torch.float32)
        y_va = torch.as_tensor(y_val,   dtype=torch.int64)

        train_ds = TensorDataset(x_tr, y_tr)
        val_ds   = TensorDataset(x_va, y_va)
        bs_tr    = min(self.batch_size, len(train_ds))
        bs_va    = min(4096, len(val_ds))

        train_dl = DataLoader(train_ds, batch_size=bs_tr, shuffle=True,  drop_last=True)
        val_dl   = DataLoader(val_ds,   batch_size=bs_va, shuffle=False)

        n_batches       = len(train_dl)
        best_val        = np.inf
        best_params     = None
        patience_count  = 0
        base_lr_vals    = [pg['lr'] for pg in opt.param_groups]   # store base LRs

        for epoch in range(self.n_epochs):
            model.train()
            for step, (xb, yb) in enumerate(train_dl):
                # Compute schedule fraction
                t        = (epoch * n_batches + step) / (self.n_epochs * n_batches)
                sched    = self._lr_schedule(t)
                for pg, base in zip(opt.param_groups, base_lr_vals):
                    pg['lr'] = base * sched

                pred = model(xb.to(self.device))
                loss = criterion(pred, yb.to(self.device))
                loss.backward()
                opt.step()
                opt.zero_grad()

            # Validation
            model.eval()
            with torch.no_grad():
                val_logits = torch.cat(
                    [model(xb.to(self.device)) for xb, _ in val_dl], dim=0
                ).cpu()
                val_loss = F.cross_entropy(val_logits, y_va).item()

            if val_loss <= best_val:
                best_val        = val_loss
                best_params     = [p.detach().clone() for p in model.parameters()]
                patience_count  = 0
            else:
                patience_count  += 1

            if self.verbose and (epoch + 1) % 25 == 0:
                val_preds = val_logits.argmax(dim=1).numpy()
                ba = balanced_accuracy_score(y_val, val_preds)
                print(f"  Epoch {epoch+1:3d}/{self.n_epochs} | val_loss={val_loss:.4f} | bal_acc={ba:.4f} | patience={patience_count}/{self.patience}")

            # Early stopping
            if patience_count >= self.patience:
                if self.verbose:
                    print(f"  Early stopping at epoch {epoch+1} (patience {self.patience} reached)")
                break

        # Restore best checkpoint
        with torch.no_grad():
            for p, p_copy in zip(model.parameters(), best_params):
                p.set_(p_copy)
        self.model_ = model
        self.best_val_ = best_val
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model_.eval()
        x = torch.as_tensor(X, dtype=torch.float32)
        ds = DataLoader(TensorDataset(x), batch_size=4096, shuffle=False)
        with torch.no_grad():
            logits = torch.cat([self.model_(xb[0].to(self.device)) for xb in ds], dim=0)
        return torch.softmax(logits, dim=1).cpu().numpy()

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(axis=1)


print("RealMLPTrainer defined.")

RealMLPTrainer defined.


## Training: 5-Fold Stratified Cross-Validation

Strategy:
- **5-fold StratifiedKFold** to preserve class balance in each fold
- Fit preprocessing pipeline on train fold, transform both train & val
- Train RealMLP with hidden_dim=512, 4 layers, 256 epochs
- Store OOF (out-of-fold) predictions + test predictions
- Final test prediction = average of 5 fold probabilities

In [15]:
import time

# ── Hyperparameters ─────────────────────────────────────────────────────────
# OPTIMIZED: Reduced model size & epochs with early stopping for 3-4x speedup
HP = dict(
    hidden_dim      = 256,     # Reduced from 512 → 256
    n_layers        = 3,       # Reduced from 4 → 3
    n_epochs        = 150,     # Reduced from 256 → 150 (early stopping helps)
    batch_size      = 4096,    # Increased from 2048 → 4096 (faster training)
    base_lr         = 0.04,
    label_smoothing = 0.1,
    dropout         = 0.0,
    device          = DEVICE,
    patience        = 25,      # NEW: Early stopping patience
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_probs  = np.zeros((len(y_all), 3))
test_probs = np.zeros((len(X_test), 3))
fold_scores = []

total_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
    fold_start = time.time()
    print(f"\n{'='*55}")
    print(f" FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*55}")

    X_tr, X_va = X_all.iloc[tr_idx], X_all.iloc[va_idx]
    y_tr, y_va = y_all[tr_idx],      y_all[va_idx]

    # ── Preprocessing: fit on train fold only ──────────────────────────────
    prep = get_realmlp_pipeline()
    prep.fit(X_tr)
    X_tr_prep  = prep.transform(X_tr).astype(np.float32)
    X_va_prep  = prep.transform(X_va).astype(np.float32)
    X_te_prep  = prep.transform(X_test).astype(np.float32)
    input_dim  = X_tr_prep.shape[1]

    print(f" Train size: {len(X_tr_prep)} | Val size: {len(X_va_prep)} | Input dim: {input_dim}")

    # ── Train RealMLP ─────────────────────────────────────────────────────
    seed_everything(SEED + fold)
    trainer = RealMLPTrainer(input_dim=input_dim, output_dim=3, verbose=True, **HP)
    trainer.fit(X_tr_prep, y_tr, X_va_prep, y_va)

    # ── OOF predictions ───────────────────────────────────────────────────
    oof_probs[va_idx] = trainer.predict_proba(X_va_prep)
    test_probs        += trainer.predict_proba(X_te_prep) / N_FOLDS

    # ── Fold score ────────────────────────────────────────────────────────
    va_preds = oof_probs[va_idx].argmax(axis=1)
    score    = balanced_accuracy_score(y_va, va_preds)
    fold_scores.append(score)
    elapsed = time.time() - fold_start
    print(f" Fold {fold+1} Balanced Accuracy: {score:.5f}  [{elapsed:.1f}s]")

# ── Final OOF Score ─────────────────────────────────────────────────────────
oof_preds = oof_probs.argmax(axis=1)
overall   = balanced_accuracy_score(y_all, oof_preds)
total_time = time.time() - total_start

print(f"\n{'='*55}")
print(f" OOF Balanced Accuracy (each fold): {[f'{s:.5f}' for s in fold_scores]}")
print(f" Mean: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")
print(f" Overall OOF: {overall:.5f}")
print(f" Benchmark:   {BENCHMARK:.5f}")
print(f" Beat benchmark: {'✓ YES' if overall >= BENCHMARK else '✗ NO'}")
print(f" Total training time: {total_time/60:.1f} min")
print(f"{'='*55}")


 FOLD 1/5
 Train size: 461877 | Val size: 115470 | Input dim: 52


KeyboardInterrupt: 

## Generating Submission

In [ ]:
# Convert averaged probabilities to class labels
test_class_ids = test_probs.argmax(axis=1)
sub_df['class'] = [INV_LABEL_MAP[c] for c in test_class_ids]

print("Test class distribution:")
print(pd.Series(sub_df['class']).value_counts())

sub_df.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv")
sub_df.head(10)

In [ ]:
# ── OOF Confusion Analysis ──────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

labels = ['GALAXY', 'QSO', 'STAR']
y_all_labels   = [INV_LABEL_MAP[c] for c in y_all]
oof_pred_labels = [INV_LABEL_MAP[c] for c in oof_preds]

print("\nClassification Report (OOF):")
print(classification_report(y_all_labels, oof_pred_labels, target_names=labels))

print("\nConfusion Matrix (OOF):")
cm = confusion_matrix(y_all_labels, oof_pred_labels, labels=labels)
cm_df = pd.DataFrame(cm, index=[f'True_{l}' for l in labels],
                          columns=[f'Pred_{l}' for l in labels])
print(cm_df)
print(f"\nFinal OOF Balanced Accuracy: {overall:.5f}")